In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import sys
from PIL import Image


In [ ]:
data_path = 'C:\\desktop 2\\project\\datasets\\acdc'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import glob
def dataFromPath(dataPath):
  path = data_path + dataPath
  images = {}
  for file in glob.glob(path):
      filename = file.split("\\")[-1]            # get the name of the .jpg file
      img = np.asarray(Image.open(file))        # read the image as a numpy array
      img =  np.array(img)/255 - 0.5
      images[filename] = img[:, :, :3]          # remove the alpha channel

  database = []

  # Populate train_data array
  for  (file_name, image) in images.items():
      id, pair_number, shoe_side, sex = file_name.split('_')
      person_index = int(id[1:]) - 1
      if shoe_side == 'left':
        shoe_side_index = 0
      else:
        shoe_side_index = 1
      pair_index = int(pair_number) - 1
      database.append([person_index,[pair_index, shoe_side_index],np.array(image)])

  database = sorted(database,key=lambda x: x[0])

  person_num = len(images)/6

  fin_data = np.zeros((int(person_num),3,2,224,224,3))
  for i,obj in enumerate(database):
    fin_data[i//6,obj[1][0],obj[1][1]] = obj[2]

  return fin_data

train_data = dataFromPath("data/train/*.jpg")
valid_data = train_data[90:]
train_data = train_data[:90]
test_m_data = dataFromPath("data/validation/*.jpg")
test_w_data = dataFromPath("data/test/*.jpg")

print("Shape of: Train - ", train_data.shape, "Validation - ", valid_data.shape)

In [ ]:
# you can change the signature and structure of this function as you please
# the code and comments below are only a suggestion to get you started
from torch.optim.lr_scheduler import ReduceLROnPlateau

def train_model(
    model,
    train_data,
    validation_data,
    batch_size=20,
    learning_rate=0.001,
    weight_decay=1e-3,
    epochs=30,
    checkpoint_path=None
):
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)
    
    best_valid_loss = float('inf')
    best_valid_acc = 0
    loss_data = []
    
    print(f"Training for Model {model.__class__.__name__}")
    print(f"n: {model.n}, Learning Rate: {learning_rate}, Weight Decay: {weight_decay}, "
          f"Epochs: {epochs}, Batch Size: {batch_size}")
    
    for epoch in range(epochs):
        model.train()
        
        positive_pair = generate_same_pair(train_data)
        negative_pair = generate_different_pair(train_data)
        
        reindex = np.random.permutation(len(negative_pair))
        negative_pair = negative_pair[reindex]
        reindex = np.random.permutation(len(positive_pair))
        positive_pair = positive_pair[reindex]
        
        for i in range(0, len(positive_pair), batch_size // 2):
            print(f"\r{epoch} {int(100 * i / len(positive_pair))}%", end='')
            
            pos_batch = positive_pair[i:i + batch_size // 2]
            neg_batch = negative_pair[i:i + batch_size // 2]
            
            if len(pos_batch) < batch_size // 2:
                continue
            
            batch = np.concatenate([pos_batch, neg_batch], axis=0)
            labels = np.concatenate([np.ones(len(pos_batch)), np.zeros(len(neg_batch))])
            
            reindex = np.random.permutation(len(batch))
            batch = batch[reindex]
            labels = labels[reindex]
            
            data = torch.Tensor(batch).permute(0, 3, 1, 2).to(device)
            labels = torch.Tensor(labels).long().to(device)
            
            # Training step
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, labels)
            loss.backward()
            
            # gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.5)
            
            optimizer.step()
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_positive_pair = generate_same_pair(validation_data)
            val_negative_pair = generate_different_pair(validation_data)
            
            val_inputs = np.concatenate([val_positive_pair, val_negative_pair], axis=0)
            val_inputs = torch.Tensor(val_inputs).permute(0, 3, 1, 2).to(device)
            num_val = len(val_positive_pair)
            val_labels = np.concatenate([np.ones(num_val), np.zeros(num_val)])
            val_labels = torch.Tensor(val_labels).long().to(device)
            
            pos_valid_accuracy, neg_valid_accuracy = get_accuracy(model, validation_data, batch_size=15, device=device)
            pos_train_accuracy, neg_train_accuracy = get_accuracy(model, train_data, batch_size=15, device=device)
            
            valid_loss = criterion(model(val_inputs), val_labels).item()
            valid_acc = 100 * (0.5 * (pos_valid_accuracy + neg_valid_accuracy))
            train_accuracy = 100 * (0.5 * (pos_train_accuracy + neg_train_accuracy))
            
            loss_data.append([epoch, valid_loss, loss.item()])
            
            print(
                f'\rEpoch [{epoch + 1}/{epochs}], '
                f'Val Loss: {valid_loss:.6f}, '
                f'Train Acc: {train_accuracy:.2f}, '
                f'Pos Val Acc: {pos_valid_accuracy*100:.2f}, '
                f'Neg Val Acc: {neg_valid_accuracy*100:.2f}, '
                f'Val Acc: {valid_acc:.2f}, '
                f'LR: {optimizer.param_groups[0]["lr"]:.8f}'
            )
            
            # Checkpoint
            if best_valid_acc == valid_acc and valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                name = model.__class__.__name__
                if checkpoint_path:
                    torch.save(model.state_dict(), 
                             f"{checkpoint_path}\\training_best_{name}.pk")
            
            if best_valid_acc < valid_acc:
                best_valid_acc = valid_acc
                best_valid_loss = valid_loss
                name = model.__class__.__name__
                if checkpoint_path:
                    torch.save(model.state_dict(), 
                             f"{checkpoint_path}\\training_best_{name}.pk")
            
            # Update learning rate
            scheduler.step(valid_loss)
    
    print(f'\nBest validation loss: {best_valid_loss:.6f}, '
          f'Best validation accuracy: {best_valid_acc:.2f}')
    
    model.load_state_dict(torch.load(f"{checkpoint_path}\\training_best_{name}.pk"))
    
    return best_valid_loss, best_valid_acc
